# 05 - Model Explainability
## RailOpt AI: Railway Maintenance Priority Prediction

## Section 1: Project Introduction

Railway maintenance decisions affect safety, service continuity, and limited engineering resources. Explainable AI (XAI) helps engineers understand which observed asset and operating conditions influenced a predicted priority, check whether the reasoning is plausible, and identify cases that need expert review.

A prediction is not a maintenance instruction. Engineers should be able to challenge explanations, verify source data, and combine model evidence with inspection standards and operational context. This notebook explains a previously trained model; it does not fit or retrain any estimator.

---
## Section 2: Import Libraries
Import visualization, data, model persistence, and SHAP explainability tools. SHAP must be installed in the active environment (`pip install shap`).

In [ ]:
from ast import literal_eval
from pathlib import Path
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")

current_dir = Path.cwd()
PROJECT_ROOT = current_dir if (current_dir / "data").exists() else current_dir.parent
MODEL_PATH = PROJECT_ROOT / "outputs" / "models" / "best_model.pkl"
FEATURE_COLUMNS_PATH = PROJECT_ROOT / "outputs" / "models" / "feature_columns.pkl"
TEST_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "test.csv"
REPORTS_DIR = PROJECT_ROOT / "outputs" / "reports"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TARGET = "priority_score"
RANDOM_STATE = 42
print(f"Project root: {PROJECT_ROOT}")
print(f"SHAP version: {shap.__version__}")

---
## Section 3: Load Model and Test Data
Load the serialized Stage 4 estimator, its feature schema, and the processed test split. Nothing in this section fits a model.

In [ ]:
required_paths = [MODEL_PATH, FEATURE_COLUMNS_PATH, TEST_DATA_PATH]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        "Required Stage 4 artifacts are missing: "
        + ", ".join(missing_paths)
        + ". Run notebooks/04_model_training.ipynb successfully first."
    )

best_model = joblib.load(MODEL_PATH)
feature_columns = joblib.load(FEATURE_COLUMNS_PATH)
test_df = pd.read_csv(TEST_DATA_PATH)
if TARGET not in test_df.columns:
    raise KeyError(f"'{TARGET}' is missing from the test dataset.")
missing_features = sorted(set(feature_columns) - set(test_df.columns))
if missing_features:
    raise KeyError(f"Test dataset is missing model features: {missing_features}")

X_test = test_df.loc[:, feature_columns].copy()
y_test = test_df[TARGET].copy()
if X_test.empty or y_test.isna().any():
    raise ValueError("Test features are empty or the target contains missing values.")
print(f"Loaded model: {MODEL_PATH.name}")
print(f"Test rows: {len(test_df):,}; model features: {len(feature_columns)}")

---
## Section 4: Model Performance Review
Prefer the final held-out test metrics saved by Stage 4. If that report is unavailable or does not contain parseable metrics, calculate the same metrics from the loaded model's predictions on this test set; this is inference only, not retraining.

In [ ]:
TRAINING_REPORT_PATH = REPORTS_DIR / "model_training_report.csv"
saved_metrics = None
if TRAINING_REPORT_PATH.exists():
    training_report = pd.read_csv(TRAINING_REPORT_PATH)
    metric_rows = training_report.loc[training_report["Item"] == "Final test metrics", "Details"]
    if not metric_rows.empty:
        try:
            saved_metrics = literal_eval(str(metric_rows.iloc[0]))
        except (ValueError, SyntaxError):
            saved_metrics = None

test_predictions = best_model.predict(X_test)
test_mse = mean_squared_error(y_test, test_predictions)
computed_metrics = {
    "R2": float(r2_score(y_test, test_predictions)),
    "RMSE": float(np.sqrt(test_mse)),
    "MAE": float(mean_absolute_error(y_test, test_predictions)),
}
metric_names = ("R2", "RMSE", "MAE")
performance_metrics = (
    {name: float(saved_metrics[name]) for name in metric_names if name in saved_metrics}
    if isinstance(saved_metrics, dict) and all(name in saved_metrics for name in metric_names)
    else computed_metrics
)
metric_source = "Stage 4 saved evaluation report" if performance_metrics is not computed_metrics else "Recomputed from the loaded model on test.csv"
print(f"Metric source: {metric_source}")
display(pd.DataFrame([performance_metrics], index=["Test"]).style.format("{:.4f}"))

---
## Section 5: Global Feature Importance
The model's tree-based importance summarizes how often and how effectively features were used to reduce training error. Large values identify influential model inputs, not causal factors; correlated variables may divide importance. The chart ranks the top 20 features.

In [ ]:
if not hasattr(best_model, "feature_importances_"):
    raise AttributeError("The loaded estimator does not expose tree feature_importances_.")
importance_df = pd.DataFrame(
    {"Feature": feature_columns, "Importance": best_model.feature_importances_}
).sort_values("Importance", ascending=False)
top_20_importance = importance_df.head(20).sort_values("Importance")
fig, ax = plt.subplots(figsize=(10, 8))
sns.barplot(data=top_20_importance, x="Importance", y="Feature", color="#287C8E", ax=ax)
ax.set(title="Top 20 Global Feature Importances", xlabel="Model importance", ylabel="")
fig.tight_layout()
importance_path = FIGURES_DIR / "global_feature_importance_top20.png"
fig.savefig(importance_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved: {importance_path}")
display(importance_df.head(20).reset_index(drop=True))
most_influential_features = importance_df.head(5)["Feature"].tolist()
least_influential_features = importance_df.tail(5).sort_values("Importance")["Feature"].tolist()

---
## Section 6: SHAP Summary Plot
SHAP values express each feature's contribution relative to the model's expected output. A positive SHAP value pushes the predicted priority upward; a negative value pushes it downward. High absolute SHAP values indicate larger influence for an observation, while low absolute values indicate smaller influence. Point color shows low-to-high feature values, not good-to-bad condition.

TreeSHAP uses the fitted model and a reproducible sample of the held-out test set; no model is retrained.

In [ ]:
EXPLANATION_SAMPLE_SIZE = min(500, len(X_test))
X_explain = X_test.sample(n=EXPLANATION_SAMPLE_SIZE, random_state=RANDOM_STATE)
explainer = shap.TreeExplainer(best_model)
shap_explanation = explainer(X_explain)
shap_values = np.asarray(shap_explanation.values)
if shap_values.ndim == 3 and shap_values.shape[-1] == 1:
    shap_values = shap_values[..., 0]
if shap_values.ndim != 2 or shap_values.shape[1] != len(feature_columns):
    raise ValueError(f"Unexpected SHAP output shape: {shap_values.shape}")

fig = plt.figure(figsize=(11, 8))
shap.summary_plot(shap_values, X_explain, max_display=20, show=False)
plt.title("SHAP Summary: Feature Contributions to Priority Score")
plt.tight_layout()
summary_plot_path = FIGURES_DIR / "shap_summary.png"
plt.savefig(summary_plot_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved: {summary_plot_path}")

mean_absolute_shap = pd.Series(
    np.abs(shap_values).mean(axis=0), index=feature_columns, name="Mean absolute SHAP"
).sort_values(ascending=False)
display(mean_absolute_shap.head(10).to_frame())

---
## Section 7: SHAP Dependence Plots
Each dependence plot relates a feature's observed value to its SHAP contribution. An upward contribution indicates the feature tends to raise predicted priority in that region; a downward contribution tends to lower it. Vertical spread can reflect interactions with other inputs. These patterns are model associations, not causal effects. The five requested features are checked against the saved model schema before plotting.

In [ ]:
dependence_features = [
    "asset_age_years",
    "defect_count",
    "inspection_score",
    "train_frequency",
    "predicted_failure_probability",
]
missing_dependence_features = sorted(set(dependence_features) - set(feature_columns))
if missing_dependence_features:
    raise KeyError(f"Requested SHAP dependence features are absent: {missing_dependence_features}")

for feature_name in dependence_features:
    plt.figure(figsize=(8, 6))
    shap.dependence_plot(
        feature_name, shap_values, X_explain, interaction_index="auto", show=False
    )
    plt.title(f"SHAP Dependence: {feature_name}")
    plt.tight_layout()
    figure_path = FIGURES_DIR / f"shap_dependence_{feature_name}.png"
    plt.savefig(figure_path, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"Saved: {figure_path}")

---
## Section 8: Explain Individual Predictions
Select five test records reproducibly. Each waterfall decomposes that record's prediction from the model's expected value: red/rightward contributions increase priority, blue/leftward contributions decrease it, and wider bars have larger contributions. SHAP explains the model output, not an engineering cause.

In [ ]:
individual_rows = X_test.sample(n=min(5, len(X_test)), random_state=RANDOM_STATE)
individual_explanations = explainer(individual_rows)
individual_predictions = best_model.predict(individual_rows)
individual_actuals = y_test.loc[individual_rows.index]
individual_summary = pd.DataFrame(
    {"Actual priority": individual_actuals.to_numpy(),
     "Predicted priority": individual_predictions},
    index=individual_rows.index,
).rename_axis("Test row")
display(individual_summary.style.format("{:.2f}"))

for position, row_index in enumerate(individual_rows.index):
    plt.figure(figsize=(10, 7))
    shap.plots.waterfall(
        individual_explanations[position], max_display=15, show=False
    )
    plt.title(
        f"Test row {row_index}: actual {individual_actuals.loc[row_index]:.1f}, "
        f"predicted {individual_predictions[position]:.1f}"
    )
    plt.tight_layout()
    waterfall_path = FIGURES_DIR / f"shap_waterfall_test_row_{row_index}.png"
    plt.savefig(waterfall_path, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"Saved: {waterfall_path}")

---
## Section 9: Prediction Confidence
This regressor does not provide calibrated confidence intervals, so SHAP magnitude must not be presented as confidence. As a limited empirical proxy, review test records with smaller absolute prediction errors as better-matched examples and larger errors as cases needing closer review. Uncertainty can arise from sparse training examples, noisy inspections, unseen routes or asset conditions, distribution shift, and omitted operational context. These error bands describe this test sample only, not future guarantees.

In [ ]:
test_error_review = pd.DataFrame(
    {"Actual priority": y_test, "Predicted priority": test_predictions},
    index=y_test.index,
)
test_error_review["Residual"] = test_error_review["Predicted priority"] - test_error_review["Actual priority"]
test_error_review["Absolute error"] = test_error_review["Residual"].abs()
print("Smallest absolute errors in this held-out sample (not calibrated confidence):")
display(test_error_review.nsmallest(5, "Absolute error"))
print("Largest absolute errors to prioritize for engineering review:")
display(test_error_review.nlargest(5, "Absolute error"))

---
## Section 10: Business Interpretation
Maintenance engineers can use global importance to identify recurring model signals and local waterfalls to inspect why a particular asset received its score. Planners can document a proposed priority with the contributing conditions, compare it with inspection evidence, and route surprising or high-error cases for review. This makes model behavior more auditable and supports trust through transparency, not by replacing engineering judgment or safety rules.

---
## Section 11: Model Limitations
The source data is synthetic and may not reproduce real railway measurement noise, maintenance practices, route variation, or rare safety events. Test-set explanations and errors do not guarantee performance on new regions, time periods, or asset populations. Tree importance can be biased by feature representation, and SHAP attributions depend on the model and background assumptions; correlated inputs complicate attribution.

Future work should validate on real, time-separated and route-separated data, monitor drift and subgroup errors, involve maintenance engineers in explanation review, and add calibrated prediction intervals or conformal uncertainty estimates before operational deployment.

---
## Section 12: Explainability Report
Compile influential and low-influence features, observed SHAP patterns, business interpretation, and recommendations into a Markdown report under `outputs/reports/`.

In [ ]:
mean_shap_order = mean_absolute_shap.sort_values(ascending=False)
top_lines = [
    f"- `{name}`: mean |SHAP| = {value:.4f}"
    for name, value in mean_shap_order.head(10).items()
]
low_lines = [
    f"- `{name}`: mean |SHAP| = {value:.4f}"
    for name, value in mean_shap_order.tail(10).sort_values().items()
]
business_features = ", ".join(
    f"`{name}`" for name in mean_shap_order.head(5).index
)

report_sections = [
    "# RailOpt AI Model Explainability Report",
    "",
    "## Model and Evaluation",
    f"- Model: {type(best_model).__name__}",
    f"- Test R²: {performance_metrics['R2']:.4f}",
    f"- Test RMSE: {performance_metrics['RMSE']:.4f}",
    f"- Test MAE: {performance_metrics['MAE']:.4f}",
    f"- Metrics source: {metric_source}",
    "",
    "## Most Influential Features",
    *top_lines,
    "",
    "## Least Influential Features",
    *low_lines,
    "",
    "## Interesting Patterns",
    "- Positive SHAP values push predictions above the model's expected priority.",
    "- Negative SHAP values push predictions below the model's expected priority.",
    "- Larger absolute SHAP values indicate greater influence for that record, not greater certainty.",
    "",
    "## Business Insights",
    f"The largest average absolute SHAP signals are {business_features}. Engineers should verify these drivers against inspection records and operating constraints.",
    "",
    "## Recommendations",
    "- Review local waterfall plots for high-priority and high-error records.",
    "- Treat attributions as decision support, not causal proof or a replacement for safety policy.",
    "- Validate on real, time-separated railway data and monitor subgroup performance and drift.",
    "- Do not interpret point predictions or SHAP values as calibrated confidence.",
]
report_text = "\n".join(report_sections) + "\n"
report_path = REPORTS_DIR / "model_explainability_report.md"
report_path.write_text(report_text, encoding="utf-8")
print(f"Saved explainability report: {report_path}")
print(f"Global tree importance top features: {importance_df.head(10)['Feature'].tolist()}")
print("All generated figures are saved under outputs/figures/.")
print("No model training or retraining was performed.")